# 📘 Git & GitHub for Data Engineering
## Version Control, Branching, Collaboration & CI/CD Foundations

**What you'll learn:**
- Why version control is essential for data engineering
- Git fundamentals (commits, branches, merges, rebases)
- Branching strategies for DE teams (trunk-based, GitFlow)
- GitHub workflows (PRs, code reviews, issues)
- What to version control (and what NOT to)
- Databricks Repos integration
- .gitignore for DE projects
- Collaboration patterns for pipeline development
- Git for SQL, configs, and pipeline code

**Prerequisites:** Notebook 02 (Python Foundations)
**Estimated Time:** 4-5 hours

---

> ⚠️ Git CLI commands cannot run directly in Databricks notebooks.
> This notebook teaches concepts + shows exact commands in markdown,
> with runnable Python cells that implement version-control concepts.

---

---
# 📗 Section 1: Why Git Matters for Data Engineering

## The Problem Without Version Control

Without Git, data engineering teams face:

| Problem | Consequence |
|---------|-------------|
| "Which version is in production?" | Deploy wrong code, break pipelines |
| "Who changed the SQL last week?" | Can't debug, can't audit |
| "How do I roll back?" | Manual recovery, data loss risk |
| "Two people editing same file" | Overwrite each other's work |
| "Where's the config for staging?" | Environment confusion |

## What Data Engineers Version Control

| ✅ Version Control | ❌ Do NOT Version Control |
|-------------------|--------------------------|
| Pipeline code (Python, SQL) | Data files (use Delta Lake) |
| Configuration (YAML, JSON) | Credentials/secrets |
| Schema definitions (DDL) | Large binaries (>100MB) |
| Quality rules & tests | Generated files (.pyc) |
| DABs bundle files | Notebook checkpoints |
| dbt models & macros | .ipynb_checkpoints/ |
| Documentation (README) | __pycache__/ |

## Git in the Data Engineering Workflow

```
Developer writes code → Git commit → Push to GitHub → PR review
→ Merge to main → CI/CD triggers → Deploy to Databricks → Pipeline runs
```

In [ ]:
# What a DE project looks like in Git
de_project_structure = {
    "my_data_pipeline/": {
        "README.md": "Project documentation",
        ".gitignore": "Files to exclude from Git",
        "databricks.yml": "DABs bundle configuration",
        "src/": {
            "pipelines/": {
                "ingest_orders.py": "Bronze layer ingestion",
                "transform_silver.py": "Silver layer cleaning",
                "build_gold.py": "Gold layer aggregation",
            },
            "sql/": {
                "create_tables.sql": "DDL for table creation",
                "quality_checks.sql": "Data quality assertions",
            },
            "utils/": {
                "config.py": "Configuration management",
                "logging_setup.py": "Logging configuration",
            }
        },
        "tests/": {
            "test_transforms.py": "Unit tests for transformations",
            "test_quality.py": "Data quality test suite",
        },
        "configs/": {
            "dev.yml": "Development environment config",
            "staging.yml": "Staging environment config",
            "prod.yml": "Production environment config",
        },
        "docs/": {
            "architecture.md": "System architecture",
            "runbook.md": "Operational runbook",
        }
    }
}

def print_tree(structure, indent=0):
    for key, value in structure.items():
        if isinstance(value, dict):
            print("  " * indent + f"📁 {key}")
            print_tree(value, indent + 1)
        else:
            print("  " * indent + f"📄 {key} — {value}")

print("📂 TYPICAL DATA ENGINEERING PROJECT STRUCTURE")
print("=" * 60)
print_tree(de_project_structure)


---
# 📗 Section 2: Git Fundamentals

## The Three Areas

```
┌─────────────────┐    git add    ┌─────────────────┐    git commit    ┌─────────────────┐
│ WORKING         │ ────────────▶ │ STAGING AREA    │ ────────────────▶│ REPOSITORY      │
│ DIRECTORY       │               │ (Index)         │                  │ (History)       │
│                 │               │                 │                  │                 │
│ Your files as   │               │ Changes ready   │                  │ Permanent       │
│ you edit them   │               │ to be committed │                  │ snapshots       │
└─────────────────┘               └─────────────────┘                  └─────────────────┘
                    ◀────────────
                      git restore
```

## Essential Git Commands

| Command | What It Does | When to Use |
|---------|-------------|-------------|
| `git init` | Create new repository | Starting a new project |
| `git clone <url>` | Copy remote repo locally | Joining an existing project |
| `git status` | Show changed files | Before every commit |
| `git add <file>` | Stage changes | After editing files |
| `git commit -m "msg"` | Save snapshot | After staging changes |
| `git push` | Upload to remote | Share your work |
| `git pull` | Download from remote | Get team's changes |
| `git branch <name>` | Create branch | Starting new feature |
| `git checkout <branch>` | Switch branch | Moving between features |
| `git merge <branch>` | Combine branches | Integrating feature |
| `git log --oneline` | View history | Understanding changes |
| `git diff` | Show changes | Before committing |
| `git stash` | Save work temporarily | Switching context |

## Commit Messages for Data Engineering

```
Good commit messages:
✅ "feat: add incremental loading for orders pipeline"
✅ "fix: handle NULL customer_ids in silver transform"
✅ "refactor: extract common date parsing to utils"
✅ "test: add quality checks for revenue calculations"
✅ "docs: update runbook with new alert procedures"

Bad commit messages:
❌ "fix stuff"
❌ "WIP"
❌ "asdfgh"
❌ "update"
```

In [ ]:
# Implement a mini version tracker (simulating Git concepts)
import hashlib
import json
from datetime import datetime

class MiniGit:
    """A simplified Git implementation to understand core concepts."""
    
    def __init__(self, repo_name):
        self.repo_name = repo_name
        self.commits = []
        self.staging = {}
        self.working_dir = {}
        self.branches = {"main": []}
        self.current_branch = "main"
        self.head = None
    
    def _hash_content(self, content):
        """SHA-1 hash (like real Git)."""
        return hashlib.sha1(content.encode()).hexdigest()[:8]
    
    def add(self, filename, content):
        """Stage a file (git add)."""
        self.working_dir[filename] = content
        self.staging[filename] = content
        print(f"   📥 Staged: {filename}")
    
    def commit(self, message):
        """Create a commit (git commit)."""
        if not self.staging:
            print("   ⚠️ Nothing to commit (staging area empty)")
            return
        
        # Create commit object
        commit_data = json.dumps(self.staging, sort_keys=True)
        commit_hash = self._hash_content(commit_data + message + str(datetime.now()))
        
        commit = {
            "hash": commit_hash,
            "message": message,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "files": dict(self.staging),
            "parent": self.head
        }
        
        self.commits.append(commit)
        self.branches[self.current_branch].append(commit_hash)
        self.head = commit_hash
        self.staging = {}
        
        print(f"   ✅ Committed: [{commit_hash}] {message}")
        return commit_hash
    
    def log(self, n=5):
        """Show commit history (git log)."""
        print(f"\n📜 COMMIT HISTORY ({self.current_branch}):")
        for commit in reversed(self.commits[-n:]):
            print(f"   {commit['hash']} — {commit['message']} ({commit['timestamp']})")
            print(f"            Files: {list(commit['files'].keys())}")
    
    def branch(self, name):
        """Create a new branch (git branch)."""
        self.branches[name] = list(self.branches[self.current_branch])
        print(f"   🌿 Created branch: {name}")
    
    def checkout(self, branch_name):
        """Switch branch (git checkout)."""
        if branch_name in self.branches:
            self.current_branch = branch_name
            print(f"   🔀 Switched to branch: {branch_name}")
        else:
            print(f"   ❌ Branch not found: {branch_name}")
    
    def diff(self):
        """Show unstaged changes (git diff)."""
        changes = {}
        for f, content in self.working_dir.items():
            if f in self.staging:
                if self.staging[f] != content:
                    changes[f] = "modified"
            else:
                changes[f] = "new"
        return changes


# Demo: Simulate a DE workflow with Git
print("🔧 MINI-GIT DEMO — Data Engineering Workflow")
print("=" * 60)

repo = MiniGit("techmart-pipeline")

# Initial commit
repo.add("src/ingest_orders.py", "def ingest(): spark.read.format('csv')...")
repo.add("src/transform.py", "def clean(): df.dropna()...")
repo.add("configs/prod.yml", "database: techmart_dw\nschedule: daily")
repo.commit("feat: initial pipeline setup")

# Feature branch
repo.branch("feature/add-quality-checks")
repo.checkout("feature/add-quality-checks")

repo.add("tests/test_quality.py", "def test_no_nulls(): assert df.filter(col('id').isNull()).count() == 0")
repo.commit("test: add null check for order_id")

repo.add("src/transform.py", "def clean(): df.dropna().dropDuplicates()...")
repo.commit("feat: add deduplication to silver transform")

# Show history
repo.log()

print("\n💡 This is exactly how real Git works:")
print("   1. Create branch for feature")
print("   2. Make commits (small, focused)")
print("   3. Push branch, create PR")
print("   4. Code review → merge to main")


---
# 📗 Section 3: Branching Strategies for Data Engineering

## Trunk-Based Development (Recommended for DE)

```
main: ──●──●──●──●──●──●──●──●──●──●──●──●──●──
            \      /       \    /       \      /
feature:     ●──●──         ●──●         ●──●──
             (1-2 days)     (1 day)      (2 days)
```

**Rules:**
- Feature branches live < 2 days
- Merge to main frequently
- Main is always deployable
- Use feature flags for incomplete features

**Why it works for DE:**
- Pipelines need frequent deployment
- Small changes are easier to debug
- Reduces merge conflicts
- Faster feedback loop

## Environment Promotion

```
main branch → Deploy to DEV → Run tests → Promote to STAGING → Validate → PROD
```

| Environment | Purpose | Data | Who Deploys |
|-------------|---------|------|-------------|
| **Dev** | Development testing | Sample data | Automatic (on merge) |
| **Staging** | Pre-production validation | Production-like data | Automatic (after dev passes) |
| **Production** | Live pipelines | Real data | Manual approval required |

In [ ]:
# ============================================
# 🎯 YOUR TURN: Design a .gitignore for a DE Project
# ============================================
# What files should be EXCLUDED from version control?
# Think about: generated files, secrets, data, caches
#
# Write your .gitignore content below:


In [ ]:
# ============================================
# ✅ SOLUTION: .gitignore for Data Engineering
# ============================================
gitignore_content = """
# ===== Python =====
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
venv/
.venv/

# ===== Databricks =====
.ipynb_checkpoints/
*.ipynb_checkpoints
.databricks/
.bundle/

# ===== Data files (use Delta Lake, not Git!) =====
*.csv
*.parquet
*.json
data/
output/

# ===== Secrets & credentials =====
.env
.env.*
*.pem
*.key
credentials.json
secrets.yml

# ===== IDE =====
.vscode/
.idea/
*.swp
*.swo

# ===== OS =====
.DS_Store
Thumbs.db

# ===== Build artifacts =====
dist/
build/
*.egg-info/

# ===== Logs =====
*.log
logs/

# ===== Terraform (if using IaC) =====
.terraform/
*.tfstate
*.tfstate.backup
"""

print("📄 .gitignore FOR DATA ENGINEERING PROJECTS")
print("=" * 60)
print(gitignore_content)

print("💡 Key Principles:")
print("   1. Never commit secrets (.env, credentials)")
print("   2. Never commit data files (use Delta Lake for data versioning)")
print("   3. Never commit generated files (__pycache__, .pyc)")
print("   4. Never commit IDE settings (.vscode, .idea)")


---
# 📗 Section 4: GitHub Collaboration Workflows

## Pull Request (PR) Workflow

```
1. Create feature branch: git checkout -b feature/add-scd2
2. Make changes, commit: git commit -m "feat: implement SCD Type 2"
3. Push branch: git push -u origin feature/add-scd2
4. Create PR on GitHub (title, description, reviewers)
5. Code review (comments, suggestions, approvals)
6. CI/CD runs tests automatically
7. Merge to main (squash or merge commit)
8. Delete feature branch
9. Deploy triggers automatically
```

## PR Best Practices for Data Engineering

| Practice | Why |
|----------|-----|
| Small PRs (< 400 lines) | Easier to review, faster feedback |
| Descriptive title | "feat: add incremental CDC for orders" |
| Include test results | Show pipeline ran successfully |
| Link to ticket/issue | Traceability |
| Add screenshots of query results | Prove correctness |
| Tag relevant reviewers | Domain experts catch issues |

## Code Review Checklist for DE PRs

```
□ Does the SQL handle NULLs correctly?
□ Is the pipeline idempotent (safe to rerun)?
□ Are there quality checks after transformation?
□ Is the schema evolution handled?
□ Are secrets properly managed (not hardcoded)?
□ Does it work for edge cases (empty data, duplicates)?
□ Is the performance acceptable at production scale?
□ Are there appropriate comments/documentation?
□ Does CI/CD pass?
```

---
# 📗 Section 5: Databricks Repos Integration [CONCEPTUAL]

## What is Databricks Repos?

Databricks Repos connects your Git repository directly to Databricks:
- Clone repos into your workspace
- Pull/push changes from the Databricks UI
- Switch branches without leaving Databricks
- Notebooks stored as `.py` files (not `.ipynb`)

```
GitHub Repository                    Databricks Workspace
┌──────────────────┐                ┌──────────────────┐
│ src/             │   git clone    │ Repos/           │
│   pipeline.py    │ ◄────────────▶ │   my-project/    │
│   transform.py   │   git pull     │     pipeline.py  │
│ configs/         │   git push     │     transform.py │
│   prod.yml       │                │     prod.yml     │
└──────────────────┘                └──────────────────┘
```

## Workflow with Repos

1. Clone repo into Databricks Repos
2. Create feature branch (in Databricks or GitHub)
3. Edit notebooks/files in Databricks
4. Commit and push from Databricks UI
5. Create PR on GitHub
6. After merge, pull latest to Databricks

## Best Practice: Notebooks as Python Files

```python
# Databricks notebook source
# COMMAND ----------
# This is how Databricks stores notebooks in Git
# Each cell is separated by "# COMMAND ----------"

spark.sql("SELECT * FROM orders")

# COMMAND ----------

# Next cell starts here
df = spark.table("customers")
```

---
# 📗 Section 4: Git Workflow for Data Engineering Teams

## The DE Development Cycle

```
1. Pick a ticket (Jira/Linear): "Add incremental loading for payments table"
2. Create branch: git checkout -b feature/incremental-payments
3. Write code: src/pipelines/ingest_payments.py
4. Write tests: tests/test_payments.py
5. Run locally: python -m pytest tests/
6. Commit: git add . && git commit -m "feat: add incremental payments ingestion"
7. Push: git push -u origin feature/incremental-payments
8. Create PR: gh pr create --title "feat: incremental payments" --body "..."
9. Code review: teammate reviews, suggests changes
10. Fix review comments: git commit -m "fix: handle NULL payment_ids"
11. CI passes: automated tests + linting
12. Merge: squash merge to main
13. Deploy: CI/CD deploys to dev → staging → prod
14. Monitor: check pipeline runs successfully
```

## Branch Naming Conventions

```
feature/add-customer-scd2          ← New feature
fix/null-handling-orders           ← Bug fix
refactor/extract-date-utils        ← Code improvement (no behavior change)
test/add-quality-checks            ← Adding tests
docs/update-runbook                ← Documentation only
hotfix/prod-pipeline-failure       ← Urgent production fix
```

## Commit Message Format (Conventional Commits)

```
<type>(<scope>): <description>

Types:
  feat:     New feature (add incremental loading)
  fix:      Bug fix (handle NULL values)
  refactor: Code change that doesn't fix a bug or add feature
  test:     Adding or fixing tests
  docs:     Documentation changes
  perf:     Performance improvement
  ci:       CI/CD changes
  chore:    Maintenance (update dependencies)

Examples:
  feat(payments): add incremental CDC ingestion
  fix(orders): handle NULL customer_ids in silver transform
  refactor(utils): extract date parsing to shared module
  test(quality): add data quality assertions for gold tables
  docs(runbook): add incident response procedures
  perf(joins): switch to broadcast join for products table
```

In [ ]:
# Simulate a realistic DE Git workflow
import hashlib
from datetime import datetime, timedelta

class GitWorkflowSimulator:
    """Simulates a realistic Git workflow for a DE team."""
    
    def __init__(self):
        self.branches = {"main": []}
        self.current_branch = "main"
        self.commits = []
        self.prs = []
    
    def checkout_new_branch(self, name):
        self.branches[name] = list(self.branches[self.current_branch])
        self.current_branch = name
        return f"Switched to new branch '{name}'"
    
    def commit(self, message, files_changed):
        commit_hash = hashlib.sha1(f"{message}{datetime.now()}".encode()).hexdigest()[:7]
        commit = {
            "hash": commit_hash,
            "message": message,
            "branch": self.current_branch,
            "files": files_changed,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        }
        self.commits.append(commit)
        self.branches[self.current_branch].append(commit_hash)
        return commit
    
    def create_pr(self, title, description, reviewers):
        pr = {
            "number": len(self.prs) + 1,
            "title": title,
            "description": description,
            "branch": self.current_branch,
            "target": "main",
            "reviewers": reviewers,
            "commits": len(self.branches[self.current_branch]) - len(self.branches["main"]),
            "status": "open",
        }
        self.prs.append(pr)
        return pr
    
    def merge_pr(self, pr_number):
        pr = self.prs[pr_number - 1]
        pr["status"] = "merged"
        # Add commits to main
        branch_commits = self.branches[pr["branch"]]
        self.branches["main"].extend(branch_commits[len(self.branches["main"]):])
        return f"PR #{pr_number} merged to main"


# Simulate a sprint of DE work
sim = GitWorkflowSimulator()

print("🔧 SIMULATED DE SPRINT — Git Workflow")
print("=" * 60)

# Developer 1: Add incremental payments pipeline
print("\n👩‍💻 Developer 1: Alice")
sim.checkout_new_branch("feature/incremental-payments")
sim.commit("feat(payments): add incremental CDC ingestion", 
           ["src/pipelines/ingest_payments.py", "configs/payments.yml"])
sim.commit("test(payments): add unit tests for payment transforms",
           ["tests/test_payments.py"])
sim.commit("docs(payments): update pipeline documentation",
           ["docs/pipelines.md"])

pr1 = sim.create_pr(
    "feat: Add incremental payments ingestion",
    "Implements CDC-based incremental loading for payments table.\n"
    "- Uses watermark pattern (payment_date > last_processed)\n"
    "- Handles NULL payment_ids (quarantine)\n"
    "- Includes quality checks",
    reviewers=["bob", "charlie"]
)
print(f"   Created PR #{pr1['number']}: {pr1['title']}")
print(f"   Branch: {pr1['branch']} → {pr1['target']}")
print(f"   Commits: {pr1['commits']}")

# Developer 2: Fix a bug
sim.checkout_new_branch("fix/null-customer-orders")
sim.commit("fix(orders): handle NULL customer_ids in silver transform",
           ["src/transforms/silver_orders.py"])

pr2 = sim.create_pr(
    "fix: Handle NULL customer_ids in orders",
    "Fixes #142. NULL customer_ids were causing join failures in gold layer.",
    reviewers=["alice"]
)
print(f"\n👨‍💻 Developer 2: Bob")
print(f"   Created PR #{pr2['number']}: {pr2['title']}")

# Merge PRs
print(f"\n✅ {sim.merge_pr(2)}")  # Bug fix merged first (smaller, urgent)
print(f"✅ {sim.merge_pr(1)}")  # Feature merged after review

print(f"\n📊 Sprint Summary:")
print(f"   Total commits: {len(sim.commits)}")
print(f"   PRs merged: {sum(1 for pr in sim.prs if pr['status'] == 'merged')}")
print(f"   Branches used: {list(sim.branches.keys())}")


---
# 📗 Section 5: CI/CD for Data Pipelines

## GitHub Actions for Databricks

```yaml
# .github/workflows/deploy.yml
name: Deploy Data Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run linting
        run: |
          pip install ruff
          ruff check src/
      
      - name: Run unit tests
        run: pytest tests/ -v --tb=short
      
      - name: Validate DABs bundle
        run: databricks bundle validate --target dev

  deploy-dev:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Deploy to DEV
        run: databricks bundle deploy --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}
      
      - name: Run integration tests
        run: databricks bundle run integration_tests --target dev

  deploy-prod:
    needs: deploy-dev
    runs-on: ubuntu-latest
    environment: production  # Requires manual approval!
    steps:
      - uses: actions/checkout@v4
      
      - name: Deploy to PROD
        run: databricks bundle deploy --target prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_PROD }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_PROD }}
```

## The Deployment Pipeline

```
PR Created → Lint + Test → Review → Merge to main
                                         │
                                    Deploy to DEV (automatic)
                                         │
                                    Integration tests pass?
                                         │
                                    Deploy to STAGING (automatic)
                                         │
                                    Validation tests pass?
                                         │
                                    Deploy to PROD (manual approval)
```

In [ ]:
# CI/CD pipeline simulation
class CIPipeline:
    """Simulates a CI/CD pipeline for data engineering."""
    
    def __init__(self):
        self.stages = []
        self.status = "pending"
    
    def add_stage(self, name, steps):
        self.stages.append({"name": name, "steps": steps, "status": "pending"})
    
    def run(self):
        print("🚀 CI/CD PIPELINE EXECUTION")
        print("=" * 60)
        
        for stage in self.stages:
            print(f"\n📌 Stage: {stage['name']}")
            all_passed = True
            
            for step in stage["steps"]:
                # Simulate step execution
                passed = step.get("passes", True)
                status = "✅" if passed else "❌"
                print(f"   {status} {step['name']}")
                
                if not passed:
                    all_passed = False
                    print(f"      Error: {step.get('error', 'Unknown error')}")
                    break
            
            stage["status"] = "passed" if all_passed else "failed"
            
            if not all_passed:
                self.status = "failed"
                print(f"\n   ❌ Pipeline FAILED at stage: {stage['name']}")
                return False
        
        self.status = "passed"
        print(f"\n✅ Pipeline PASSED — all {len(self.stages)} stages successful!")
        return True


# Build a realistic DE CI/CD pipeline
pipeline = CIPipeline()

pipeline.add_stage("Lint & Format", [
    {"name": "ruff check src/", "passes": True},
    {"name": "black --check src/", "passes": True},
    {"name": "mypy src/ --ignore-missing-imports", "passes": True},
])

pipeline.add_stage("Unit Tests", [
    {"name": "pytest tests/unit/ -v", "passes": True},
    {"name": "Coverage check (>80%)", "passes": True},
])

pipeline.add_stage("Integration Tests", [
    {"name": "pytest tests/integration/ -v", "passes": True},
    {"name": "Data quality assertions", "passes": True},
])

pipeline.add_stage("Bundle Validation", [
    {"name": "databricks bundle validate --target dev", "passes": True},
])

pipeline.add_stage("Deploy to DEV", [
    {"name": "databricks bundle deploy --target dev", "passes": True},
    {"name": "Run smoke tests on DEV", "passes": True},
])

pipeline.add_stage("Deploy to PROD", [
    {"name": "Manual approval received", "passes": True},
    {"name": "databricks bundle deploy --target prod", "passes": True},
    {"name": "Post-deploy health check", "passes": True},
])

pipeline.run()


---
# 📗 Section 6: Handling Conflicts & Common Scenarios

## Merge Conflicts in Data Engineering

Conflicts happen when two people edit the same file. Common in DE:

| Scenario | How It Happens | Resolution |
|----------|---------------|------------|
| **Same SQL file** | Two people modify the same transformation | Review both changes, combine logic |
| **Config file** | Both add new pipeline configs | Merge both entries |
| **Schema change** | One adds column, other renames | Coordinate, apply in order |
| **Test file** | Both add tests for same function | Keep both tests |

## Git Commands for Conflict Resolution

```bash
# You're on feature branch, main has new commits
git checkout main
git pull                          # Get latest main
git checkout feature/my-branch
git rebase main                   # Replay your commits on top of main

# If conflicts:
# 1. Open conflicted files
# 2. Look for <<<<<<< HEAD ... ======= ... >>>>>>> markers
# 3. Choose correct version (or combine both)
# 4. git add <resolved-file>
# 5. git rebase --continue

# Alternative: merge instead of rebase
git merge main                    # Creates a merge commit
```

## Git Stash — Save Work Temporarily

```bash
# Scenario: You're mid-feature but need to fix a production bug

git stash                         # Save current work
git checkout main
git checkout -b hotfix/prod-fix
# ... fix the bug ...
git commit -m "hotfix: fix prod pipeline"
git push
# ... create PR, merge ...

git checkout feature/my-branch
git stash pop                     # Restore your saved work
# Continue where you left off!
```

In [ ]:
# ============================================
# 🎯 YOUR TURN: Design a Git Workflow
# ============================================
# Scenario: Your team of 5 data engineers needs a Git workflow.
# 
# Requirements:
# - 3 environments: dev, staging, prod
# - Daily deployments to dev (automatic)
# - Weekly deployments to prod (manual approval)
# - Hotfix capability (bypass normal flow for urgent fixes)
# - Code review required for all changes
#
# Design:
# 1. What branching strategy? (trunk-based, GitFlow, other?)
# 2. What branch protection rules?
# 3. What CI/CD stages?
# 4. How do you handle hotfixes?
#
# Write your design below:


In [ ]:
# ============================================
# ✅ SOLUTION: DE Team Git Workflow
# ============================================
workflow_design = {
    "branching_strategy": "Trunk-based development",
    "rationale": "Small team (5), frequent deployments, simple is better",
    
    "branches": {
        "main": "Production-ready code. All PRs merge here.",
        "feature/*": "Short-lived (1-2 days). One per ticket.",
        "hotfix/*": "Emergency fixes. Bypass normal review for speed.",
    },
    
    "branch_protection_rules": [
        "Require 1 approval for feature branches",
        "Require CI to pass before merge",
        "No direct pushes to main (PRs only)",
        "Squash merge (clean history)",
        "Auto-delete branch after merge",
    ],
    
    "ci_cd_stages": {
        "On PR": ["Lint", "Unit tests", "Bundle validate"],
        "On merge to main": ["Deploy to DEV", "Integration tests"],
        "Daily (scheduled)": ["Deploy to STAGING", "Full regression"],
        "Weekly (manual)": ["Deploy to PROD", "Smoke tests", "Alert on-call"],
    },
    
    "hotfix_process": [
        "1. Create hotfix/* branch from main",
        "2. Fix the issue (minimal change)",
        "3. Get 1 quick review (can be async)",
        "4. Merge to main",
        "5. Deploy directly to PROD (skip staging)",
        "6. Backfill: run full tests after deployment",
    ]
}

print("📋 DE TEAM GIT WORKFLOW DESIGN")
print("=" * 60)
print(f"\n   Strategy: {workflow_design['branching_strategy']}")
print(f"   Rationale: {workflow_design['rationale']}")

print("\n   Branches:")
for branch, purpose in workflow_design["branches"].items():
    print(f"      {branch}: {purpose}")

print("\n   Protection Rules:")
for rule in workflow_design["branch_protection_rules"]:
    print(f"      • {rule}")

print("\n   CI/CD Stages:")
for trigger, stages in workflow_design["ci_cd_stages"].items():
    print(f"      {trigger}: {' → '.join(stages)}")

print("\n   Hotfix Process:")
for step in workflow_design["hotfix_process"]:
    print(f"      {step}")


---
# 📗 Section 7: Advanced Git Patterns for Data Engineering

## Monorepo vs Multi-repo

| Approach | Pros | Cons | Best For |
|----------|------|------|----------|
| **Monorepo** | Single PR for cross-cutting changes, shared tooling | Large repo, complex CI | Small-medium teams |
| **Multi-repo** | Clear ownership, independent deployments | Cross-repo changes are hard | Large teams, microservices |

## Git Hooks for Data Engineering

```bash
# .git/hooks/pre-commit (runs before every commit)
#!/bin/bash
# Run linting
ruff check src/ || exit 1
# Run unit tests
pytest tests/unit/ -q || exit 1
# Check for hardcoded credentials
grep -r "password\s*=" src/ && echo "ERROR: Hardcoded password!" && exit 1
echo "Pre-commit checks passed!"
```

## Semantic Versioning for Data Pipelines

```
v2.1.3
  │ │ └── Patch: bug fix, no behavior change
  │ └──── Minor: new feature, backward compatible
  └────── Major: breaking change (schema change, removed column)

Examples:
  v1.0.0 → v1.0.1: Fixed NULL handling bug
  v1.0.1 → v1.1.0: Added new gold table
  v1.1.0 → v2.0.0: Renamed column (BREAKING CHANGE)
```

## Git Bisect -- Finding When a Bug Was Introduced

```bash
# Start bisect
git bisect start
git bisect bad HEAD          # Current commit is broken
git bisect good v1.2.0       # This version was working

# Git checks out a commit halfway between good and bad
# Test it, then tell git:
git bisect good   # or: git bisect bad

# Git narrows down until it finds the exact commit that broke things
# Takes log2(N) steps -- very efficient!
```

In [ ]:
# Git workflow best practices for DE teams
print("Git Best Practices for Data Engineering")
print("=" * 60)

best_practices = {
    "Commit hygiene": [
        "Commit often (small, focused commits)",
        "Each commit should be a single logical change",
        "Write meaningful commit messages (feat/fix/refactor/test/docs)",
        "Never commit secrets or credentials",
        "Never commit large data files (use Delta Lake for data)",
    ],
    "Branch strategy": [
        "Short-lived feature branches (< 2 days)",
        "Name branches descriptively: feature/add-cdc-pipeline",
        "Delete branches after merging",
        "Keep main always deployable",
        "Use feature flags for incomplete features",
    ],
    "Code review": [
        "Review for correctness AND performance",
        "Check NULL handling in SQL/PySpark",
        "Verify idempotency (safe to rerun)",
        "Check for hardcoded values (use configs)",
        "Ensure tests cover the change",
    ],
    "CI/CD": [
        "All tests must pass before merge",
        "Lint and format checks are mandatory",
        "Deploy to dev automatically on merge",
        "Require manual approval for prod",
        "Keep CI pipeline under 10 minutes",
    ]
}

for category, practices in best_practices.items():
    print(f"\n{category}:")
    for p in practices:
        print(f"  * {p}")


## Common Git Mistakes in Data Engineering

| Mistake | Consequence | Prevention |
|---------|-------------|------------|
| Committing `.env` files | Credentials exposed | Add to `.gitignore`, use pre-commit hooks |
| Committing large CSV/Parquet | Repo bloated, slow clones | Always add data files to `.gitignore` |
| Force-pushing to main | Overwrites team's work | Protect main branch (no force push) |
| Merging without tests | Broken pipeline in prod | Require CI to pass before merge |
| Vague commit messages | Can't understand history | Use conventional commits format |
| Long-lived branches | Massive merge conflicts | Merge to main at least daily |

---
*Notebook 05 of the Databricks Data Engineering series*
*Study Order: 5*

---
# 📗 Section 6: Summary & Quick Reference

## Git Commands Cheat Sheet

```
SETUP:
  git init                          Create new repo
  git clone <url>                   Copy remote repo

DAILY WORKFLOW:
  git status                        See what's changed
  git add <file>                    Stage changes
  git add .                         Stage all changes
  git commit -m "message"           Save snapshot
  git push                          Upload to remote
  git pull                          Download from remote

BRANCHING:
  git branch <name>                 Create branch
  git checkout <branch>             Switch branch
  git checkout -b <name>            Create + switch
  git merge <branch>                Merge into current
  git branch -d <name>              Delete branch

HISTORY:
  git log --oneline                 Compact history
  git log --graph                   Visual branch history
  git diff                          Show unstaged changes
  git diff --staged                 Show staged changes

UNDO:
  git restore <file>                Discard changes
  git restore --staged <file>       Unstage
  git revert <hash>                 Undo a commit (safe)
  git reset --hard <hash>           Reset to commit (dangerous!)

COLLABORATION:
  git remote -v                     Show remotes
  git fetch                         Download without merge
  git stash                         Save work temporarily
  git stash pop                     Restore stashed work
```

## Next Steps

- **Notebook 04:** Medallion Architecture (version-controlled pipelines)
- **Notebook 28:** CI/CD & Version Control (automation with Git)
- **Notebook 35:** Declarative Automation Bundles (deploy with Git)

---
*Notebook 43 of the Databricks Data Engineering series*
*Study Order: 5*